In [13]:
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import re

# =========================
# CONEXIÓN MONGO
# =========================
client = MongoClient("mongodb://database:27017/")
db = client["portalinmobiliario"]
collection = db["propiedades"]

# Limpiar colección
collection.delete_many({})
print("Limpieza completada")

Limpieza completada


In [14]:

# =========================
# SELENIUM
# =========================
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

print("Navegador iniciado")
print("Ver scraping en: http://localhost:6080/vnc.html")


# =========================
# SCROLL
# =========================
def scroll():

    for _ in range(5):
        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )
        time.sleep(2)


# =========================
# CIUDADES
# =========================
ciudades = {
    "Coquimbo": "https://www.toctoc.com/arriendo/departamento/coquimbo",
    "La Serena": "https://www.toctoc.com/arriendo/departamento/la-serena"
}


# =========================
# SCRAPING
# =========================
for ciudad, url in ciudades.items():

    print(f"\nScrapeando {ciudad}")

    driver.get(url)
    time.sleep(6)

    scroll()

    bloques = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'property-card')]"
    )

    print("Bloques encontrados:", len(bloques))

    for bloque in bloques:

        texto = bloque.text.lower()

        # =========================
        # PRECIO
        # =========================
        precio = "$0"

        p = re.findall(r"\$[\d\.]+", texto)
        if p:
            precio = p[0]

        # =========================
        # DORMITORIOS
        # =========================
        dormitorios = 0

        d = re.findall(r"\d+\s?dorm", texto)
        if d:
            dormitorios = int(
                re.findall(r"\d+", d[0])[0]
            )

        # =========================
        # BAÑOS
        # =========================
        banos = 0

        b = re.findall(r"\d+\s?ba", texto)
        if b:
            banos = int(
                re.findall(r"\d+", b[0])[0]
            )

        # =========================
        # GUARDAR
        # =========================
        if precio != "$0":

            data = {
                "ciudad": ciudad,
                "precio": precio,
                "dormitorios": dormitorios,
                "banos": banos
            }

            collection.insert_one(data)

            print(
                f"Guardado: {ciudad} | {precio} | {dormitorios}D {banos}B"
            )


print(
    "\nExtracción terminada:",
    collection.count_documents({}),
    "propiedades"
)

print("Datos guardados en MongoDB")
print("Ver en http://localhost:8081")

driver.quit()

Navegador iniciado
Ver scraping en: http://localhost:6080/vnc.html

Scrapeando Coquimbo
Bloques encontrados: 0

Scrapeando La Serena
Bloques encontrados: 0

Extracción terminada: 0 propiedades
Datos guardados en MongoDB
Ver en http://localhost:8081
